# Feature Extraction from Ablated Models

To uncover how different annotation dimensions influence model reasoning in IEI variant prioritization, we extract two types of interpretable features from the **last hidden states** of each ablated model during inference:

## 1. Semantic Features

- **Goal**: Capture how the model *attends to specific annotation phrases* (e.g., “GnomAD: 0.001”) when making predictions.
- **Method**:
  - Identify token positions corresponding to **predefined target phrases** (e.g., `"1000G"`, `"GnomAD"`) in the input prompt.
  - Extract their hidden states from `last_hidden_state`.
  - Compute a **weighted average** using attention weights from the **output (label) token** toward these phrase tokens.
- **Result**: A dense vector per sample per ablation model, representing *semantic focus* on that annotation type.

## 2. Effect Features

- **Goal**: Quantify the *decision-level impact* of adding one annotation dimension (e.g., allele frequency) beyond the basic context.
- **Method**:
  - Extract the hidden state of the **last token** (a fixed separator or EOS token) from:
    - The **ablated model** (e.g., `af`)
    - The **basic model** (`basic`)
  - Compute the difference:
    $$
    \text{Effect Feature} = h_{\text{last}}^{\text{(af)}} - h_{\text{last}}^{\text{(basic)}}
    $$
  - This “perturbation vector” reflects how the added annotation **shifts the model’s internal representation** at decision time.
- **Result**: A perturbation vector per sample, encoding the *functional influence* of that annotation dimension.

## 3. IG Score (Integrated Gradients Attribution)

- **Goal**: Attribute the model’s prediction to individual input tokens using **gradient-based saliency**, revealing which parts of the prompt most strongly drive the output.
- **Method**: We use **Integrated Gradients (IG)** as implemented in [`LLMGradientAttribution`](https://github.com/meta-pytorch/captum), a Captum-based toolkit for LLM interpretability.
- **Formula**: For an input token sequence \($ x \in \mathbb{R}^d $\) and baseline \($ x' $\) (e.g., all-zero or [PAD] tokens), the IG attribution for token \($ i $\) is:
  $$
  \text{IG}_i(x) = (x_i - x'_i) \times \int_{\alpha=0}^{1} \frac{\partial F(x' + \alpha (x - x'))}{\partial x_i}  d\alpha
  $$
  where \($ F(\cdot) $\) is the model’s logit output for the predicted class (e.g., “Pathogenic”).
- **Implementation**: Approximated via Riemann sum with 50 steps; baseline set to embedding of `[PAD]` token.
- **Integration**: Controlled by the `--IGscore` flag in `GetEmb.py`. When enabled, the script outputs per-token IG scores aligned with input tokens.
- **Output**: A list of floats (`ig_scores`) in the JSONL, same length as input token IDs, indicating attribution strength.


> **Note**: All features are extracted **per sample and per ablated model**, enabling downstream analysis (e.g., clustering, correlation with phenotype severity, or probing for biological signals).

The extraction is performed via the `GetEmb.py` script, which supports full hidden-state access, original feature extraction (All extracted features are raw, unaggregated metadata vectors needed to be further processed via attention weighting or differencing; see the `aggregate.py` script for implementation details.), and gradient-based attribution through Captum.

In [ ]:
import os
from pathlib import Path
CHECKPOINT_ROOT = "/path/to/ablated/model/checkpoints"
TESTSET_DIR = "/path/to/test_set/jsonl"
OUTPUT_DIR = "./extracted_features"
os.makedirs(OUTPUT_DIR, exist_ok=True)

# Define ablation models and their target phrases for semantic feature extraction
ABLATION_CONFIG = {
    "af": ["1000G", "GnomAD", "GnomAD-EAS"],
    "ACMG": ["ACMG evidence items", "pathogenicity assessment"],
    "dbsig": ["Clinvar database", "HGMD database"],
    "pred": ["SIFT", "Polyphen2-HDIV", "Polyphen2-HVAR", "MutationTaster2", "Revel", "Condel", "SpliceAI", "dbscSnv_RF", "dbscSnv_ADA", "PhyloP Vertebrates", "PhyloP Placental Mammals", "GERP++"],
    "dis": ["associated diseases"],
    "IUIS": [], # controlled by --IUIS, see GetEmb.py
    "all": ["1000G", "GnomAD", "GnomAD-EAS", "ACMG evidence items", "pathogenicity assessment", "Clinvar database", "HGMD database", "SIFT", "Polyphen2-HDIV", "Polyphen2-HVAR", "MutationTaster2", "Revel", "Condel", "SpliceAI", "dbscSnv_RF", "dbscSnv_ADA", "PhyloP Vertebrates", "PhyloP Placental Mammals", "GERP++", "associated diseases"] # all above
    # Note: 'basic' do not need target_phrases for semantic feature (optional) and effect feature is extracted by default for all models and appended as the last feature.
}

MODEL_NAMES = ["basic", "af", "ACMG", "dbsig", "pred", "dis", "IUIS", "all"]

# Generate extraction commands
def generate_getemb_command(model_name, target_phrases=None, use_IUIS=False):
    ckpt = os.path.join(CHECKPOINT_ROOT, model_name)
    if not os.path.exists(ckpt):
        print(f"Checkpoint not found: {ckpt}")
        return None

    cmd = f"""export CUDA_VISIBLE_DEVICES=3 && python GetEmb.py \\
    --model_path "{ckpt}" \\
    --input_jsonl "{TESTSET_DIR}/*.jsonl" \\
    --output_jsonl "{OUTPUT_DIR}/{model_name}.jsonl" \\
    --max_new_tokens 9000 \\
    --temperature 0 \\
    --use_fast_tokenizer"""

    if target_phrases:
        phrases_str = " ".join([f'"{p}"' for p in target_phrases])
        cmd += f" \\\n    --target_phrases {phrases_str}"

    if use_IUIS:
        cmd += " \\\n    --IUIS"

    return cmd

for model in MODEL_NAMES:
    phrases = ABLATION_CONFIG.get(model, [])
    use_IUIS = (model == "IUIS" or model == "all")
    cmd = generate_getemb_command(model, target_phrases=phrases, use_IUIS=use_IUIS)
    if cmd:
        print(cmd + "\n")

## Principal Component Analysis (PCA) of Feature Representations

To visualize the global structure of our extracted representations, we apply PCA to both effect-based and semantic-based features under two conditions:
- **Excluding the "all" category**: focuses on fine-grained, variant-specific annotations.
- **Including the "all" category**: captures the aggregate representation that pools information across all sources.

This yields four distinct visualizations:
1. Semantic features (excluding "all")
2. Semantic features (including "all")
3. Effect features (excluding "all")
4. Effect features (including "all")

Each plot shows the first two principal components, colored by feature source (e.g., `"af"`, `"clinvar"`, `"all"`), enabling assessment of:
- Cluster separation by annotation type
- Relative compactness or dispersion of representations
- The influence of the aggregate "all" embedding on overall geometry


In [ ]:
import joblib
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.decomposition import PCA
from typing import Tuple, Dict

def prepare_clustering_data(data: Dict[str, np.ndarray]) -> Tuple[np.ndarray, np.ndarray, None]:
    """
    Stack feature vectors and create corresponding labels.
    Returns:
        X: (N, D) feature matrix
        y: (N,) label array (source tags)
        scaler: placeholder (not used here; kept for API compatibility)
    """
    X_list, y_list = [], []
    for tag, vectors in data.items():
        if vectors.size > 0:
            X_list.append(vectors)
            y_list.extend([tag] * vectors.shape[0])
    X = np.vstack(X_list)
    y = np.array(y_list)
    return X, y, None


def plot_pca(X: np.ndarray, y: np.ndarray, title: str = "PCA", continuous: bool = False):
    pca = PCA(n_components=2)
    X_pca = pca.fit_transform(X)

    # Set publication-quality font and output settings
    plt.rcParams["font.family"] = "DejaVu Serif"
    plt.rcParams["pdf.fonttype"] = 42
    plt.rcParams["ps.fonttype"] = 42
    plt.rcParams["svg.fonttype"] = "none"

    # Colorblind-friendly palette (Wong, 2011)
    CB = ["#0072B2", "#D55E00", "#009E73", "#CC79A7", "#E69F00", "#56B4E9", "#F0E442"]
    sns.set_palette(CB)

    plt.figure(figsize=(8, 6))

    if continuous:
        scatter = plt.scatter(
            X_pca[:, 0], X_pca[:, 1],
            c=y, cmap='viridis', s=20, rasterized=True
        )
        plt.colorbar(scatter)
    else:
        # Preserve label order of first appearance
        label_order = list(dict.fromkeys(y))
        y_cat = pd.Categorical(y, categories=label_order, ordered=True)
        sns.scatterplot(
            x=X_pca[:, 0], y=X_pca[:, 1],
            hue=y_cat, hue_order=label_order,
            s=20, rasterized=True, legend='brief'
        )
        plt.legend(title="Feature Source", frameon=False)

    # Title with explained variance
    plt.title(
        f"{title}\n"
        f"PC1: {pca.explained_variance_ratio_[0]:.1%}, "
        f"PC2: {pca.explained_variance_ratio_[1]:.1%}",
        fontsize=14, pad=15
    )
    plt.xlabel("PC1", fontsize=12)
    plt.ylabel("PC2", fontsize=12)
    plt.grid(False)
    plt.tight_layout()

    # Save in multiple formats for publication flexibility
    for ext in ['pdf', 'svg', 'png']:
        plt.savefig(f"{title}.{ext}", dpi=300, bbox_inches='tight')
    plt.show()

In [ ]:
# Load feature dictionaries (after aggregation)
file_path_effect = 'features/features_effect.pkl'
file_path_semantic = 'features/features_semantic.pkl'

feature_effect_all = joblib.load(file_path_effect)
feature_semantic_all = joblib.load(file_path_semantic)
print(feature_effect_all.keys())
print(feature_semantic_all.keys())

In [ ]:
feature_semantic_no_all = {k: v for k, v in feature_semantic_all.items() if k != "all"}
X_sem_no, y_sem_no, _ = prepare_clustering_data(feature_semantic_no_all)
plot_pca(X_sem_no, y_sem_no, title="PCA_Semantic_Without_All")

X_sem_all, y_sem_all, _ = prepare_clustering_data(feature_semantic_all)
plot_pca(X_sem_all, y_sem_all, title="PCA_Semantic_With_All")

feature_effect_no_all = {k: v for k, v in feature_effect_all.items() if k != "all"}
X_eff_no, y_eff_no, _ = prepare_clustering_data(feature_effect_no_all)
plot_pca(X_eff_no, y_eff_no, title="PCA_Effect_Without_All")

X_eff_all, y_eff_all, _ = prepare_clustering_data(feature_effect_all)
plot_pca(X_eff_all, y_eff_all, title="PCA_Effect_With_All")